In [1]:
!pip install mediapipe==0.10.21 opencv-python pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 9.5 MB/s eta 0:00:00

In [1]:
import pandas as pd
import numpy as np
import mediapipe as mp
import cv2
import os
from google.colab import drive

# --- 1. MOUNT AND SETUP ---
try:
    drive.mount('/content/drive', force_remount=True)
except:
    print("Drive already mounted.")

# Paths
file_id = '1FtRdaCpGFbUYhKbnp-TBJQeBRFgn5y3v'
csv_url = f'https://drive.google.com/uc?export=download&id={file_id}'
video_folder = '/content/drive/MyDrive/ASL_Videos_100'
keypoints_dir = '/content/drive/MyDrive/ASL_Keypoints_Data'

os.makedirs(keypoints_dir, exist_ok=True)

# --- 2. DYNAMIC PATH VERIFICATION ---
# This checks if your files have extensions like .mp4, .mkv, etc.
sample_files = os.listdir(video_folder)
extension = ""
if sample_files:
    if "." in sample_files[0]:
        extension = "." + sample_files[0].split(".")[-1]
    print(f"Detected file extension: '{extension}'")
else:
    print("ERROR: Video folder is empty!")

# --- 3. DATA PREPARATION ---
df = pd.read_csv(csv_url, dtype={'gloss': str, 'video_id': str})
target_words = df['gloss'].unique()[:3] # First 3 words
df_filtered = df[df['gloss'].isin(target_words)]
print(f"Processing words: {target_words}")

# --- 4. EXTRACTION LOGIC ---
mp_holistic = mp.solutions.holistic

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

# --- 5. EXECUTION LOOP ---
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for index, row in df_filtered.iterrows():
        vid_id, gloss = str(row['video_id']), row['gloss']

        # Folder structure: Keypoints / Word / video_id.npy
        save_path = os.path.join(keypoints_dir, gloss)
        os.makedirs(save_path, exist_ok=True)
        npy_file = os.path.join(save_path, f"{vid_id}.npy")

        if os.path.exists(npy_file):
            continue

        # Construct path using detected extension
        video_path = os.path.join(video_folder, f"{vid_id}{extension}")
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"File NOT found: {video_path}")
            continue

        video_sequence = []
        start, end = int(row['frame_start']), int(row['frame_end'])
        if end == -1: end = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        curr = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or curr > end: break
            if curr >= start:
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)
                video_sequence.append(extract_keypoints(results))
            curr += 1

        cap.release()
        if video_sequence:
            np.save(npy_file, video_sequence)
            print(f"Successfully saved: {vid_id} ({gloss}) | Frames: {len(video_sequence)}")

print("\n--- Extraction complete for the first 3 words! ---")

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(


Mounted at /content/drive
Detected file extension: '.mp4'
Processing words: ['accident' 'africa' 'all']
Successfully saved: 00618 (accident) | Frames: 27
Successfully saved: 00624 (accident) | Frames: 108
Successfully saved: 00626 (accident) | Frames: 43
Successfully saved: 00629 (accident) | Frames: 148
Successfully saved: 00634 (accident) | Frames: 39
Successfully saved: 00635 (accident) | Frames: 98
Successfully saved: 00639 (accident) | Frames: 104
Successfully saved: 65009 (accident) | Frames: 55
Successfully saved: 01382 (africa) | Frames: 44
Successfully saved: 01383 (africa) | Frames: 145
Successfully saved: 01384 (africa) | Frames: 64
Successfully saved: 01387 (africa) | Frames: 39
Successfully saved: 01388 (africa) | Frames: 122
Successfully saved: 01393 (africa) | Frames: 71
Successfully saved: 01394 (africa) | Frames: 84
Successfully saved: 01395 (africa) | Frames: 82
Successfully saved: 01398 (africa) | Frames: 77
Successfully saved: 65029 (africa) | Frames: 65
Successfull

In [2]:
import os
import numpy as np

# Set your desired sequence length (e.g., 30 frames is standard for ASL)
SEQUENCE_LENGTH = 25

def standardize_sequences(keypoints_dir):
    for word in os.listdir(keypoints_dir):
        word_path = os.path.join(keypoints_dir, word)
        if not os.path.isdir(word_path): continue

        for npy_file in os.listdir(word_path):
            file_path = os.path.join(word_path, npy_file)
            window = np.load(file_path)

            # Scenario A: Video is too long -> Truncate
            if len(window) > SEQUENCE_LENGTH:
                standardized = window[:SEQUENCE_LENGTH]

            # Scenario B: Video is too short -> Pad with zeros
            else:
                padding = np.zeros((SEQUENCE_LENGTH - len(window), 1662))
                standardized = np.concatenate([window, padding], axis=0)

            # Overwrite the file with the new standardized version
            np.save(file_path, standardized)
            print(f"Standardized {npy_file} to {standardized.shape}")

# Run the standardization
standardize_sequences(keypoints_dir)

Standardized 00618.npy to (25, 1662)
Standardized 00624.npy to (25, 1662)
Standardized 00626.npy to (25, 1662)
Standardized 00629.npy to (25, 1662)
Standardized 00634.npy to (25, 1662)
Standardized 00635.npy to (25, 1662)
Standardized 00639.npy to (25, 1662)
Standardized 65009.npy to (25, 1662)
Standardized 01382.npy to (25, 1662)
Standardized 01383.npy to (25, 1662)
Standardized 01384.npy to (25, 1662)
Standardized 01387.npy to (25, 1662)
Standardized 01388.npy to (25, 1662)
Standardized 01393.npy to (25, 1662)
Standardized 01394.npy to (25, 1662)
Standardized 01395.npy to (25, 1662)
Standardized 01398.npy to (25, 1662)
Standardized 65029.npy to (25, 1662)
Standardized 01912.npy to (25, 1662)
Standardized 01986.npy to (25, 1662)
Standardized 01987.npy to (25, 1662)
Standardized 01988.npy to (25, 1662)
Standardized 01992.npy to (25, 1662)
Standardized 01995.npy to (25, 1662)
Standardized 01996.npy to (25, 1662)
Standardized 02003.npy to (25, 1662)
Standardized 65043.npy to (25, 1662)
S